# 07b - Forward/Backward Fill and axis=1 (Hands-On)

`loans.csv` can't correctly demonstrate either technique from `07_Filling_Missing_Values.MD`'s Method B - its rows are independent, unrelated applicants, so there's no real order to fill along, and its columns aren't comparable measurements. This notebook uses two small datasets built specifically for the job:

- `daily_temperature.csv` - a genuine ordered time series, for `ffill`/`bfill`
- `store_sales_wide.csv` - genuinely comparable columns (Mon-Fri sales), for a real `axis=1` case

In [1]:
import pandas as pd

## ffill - carry the last known reading forward

In [2]:
temps = pd.read_csv("daily_temperature.csv")
temps

,Date,Temperature_C
0,2026-01-01,24.0
1,2026-01-02,NaN
2,2026-01-03,25.6
3,2026-01-04,25.3
4,2026-01-05,25.1
5,2026-01-06,24.8
6,2026-01-07,25.1
7,2026-01-08,25.1
8,2026-01-09,NaN
9,2026-01-10,24.4


In [4]:
temps.isnull().sum()

Date             0
Temperature_C    5
dtype: int64

In [5]:
temps["Temperature_C"].ffill()

0     24.0
1     24.0
2     25.6
3     25.3
4     25.1
5     24.8
6     25.1
7     25.1
8     25.1
9     24.4
10    25.4
11    25.3
12    25.3
13    25.6
14    25.4
15    25.7
16    26.2
17    26.0
18    26.0
19    26.0
20    25.8
21    24.9
22    25.2
23    24.8
24    23.6
25    23.1
26    22.9
27    22.1
28    21.2
29    21.3
Name: Temperature_C, dtype: float64

Look at **Jan 19 and Jan 20** - two missing days back to back. Both get filled with the same number, Jan 18's `26.0`. `ffill` doesn't guess a trend, it just repeats the last thing it actually saw - worth knowing before trusting it across a multi-day gap.

## bfill - carry the next known reading backward

In [5]:
temps["Temperature_C"].bfill()

0     24.0
1     25.6
2     25.6
3     25.3
4     25.1
5     24.8
6     25.1
7     25.1
8     24.4
9     24.4
10    25.4
11    25.3
12    25.6
13    25.6
14    25.4
15    25.7
16    26.2
17    26.0
18    25.8
19    25.8
20    25.8
21    24.9
22    25.2
23    24.8
24    23.6
25    23.1
26    22.9
27    22.1
28    21.2
29    21.3
Name: Temperature_C, dtype: float64

Same two missing days now both become Jan 21's `25.8` instead of Jan 18's `26.0` - a different guess than `ffill` gave. Neither is "the truth"; they're two different assumptions about which direction to borrow from.

## Now the wrong-tool case - running ffill on loans.csv anyway

In [6]:
loans = pd.read_csv("loans.csv")
loans["Gender"].head(10)

0      Male
1      Male
2    Female
3      Male
4      Male
5    Female
6      Male
7       NaN
8      Male
9      Male
Name: Gender, dtype: str

In [7]:
loans["Gender"].ffill().head(10)

0      Male
1      Male
2    Female
3      Male
4      Male
5    Female
6      Male
7      Male
8      Male
9      Male
Name: Gender, dtype: str

Compare row 7 in both outputs - its missing `Gender` silently becomes whatever row 6 happened to be. The code runs, the gap gets filled, the output *looks* reasonable. But row 6 and row 7 are two unrelated loan applicants - there's no real-world reason #7's gender should match #6's. This is `07_Filling_Missing_Values.MD`'s warning, now shown rather than just argued: a working line of code is not the same thing as a correct one.

## A genuine axis=1 case - filling across related columns

In [8]:
sales = pd.read_csv("store_sales_wide.csv").set_index("Store_ID")
sales

,Mon_Sales,Tue_Sales,Wed_Sales,Thu_Sales,Fri_Sales
Store_ID,,,,,
Store_1,7643.0,7623.0,8217.0,7670.0,7913.0
Store_2,13102.0,13203.0,NaN,12921.0,12086.0
Store_3,11210.0,11376.0,10456.0,10688.0,10593.0
Store_4,13055.0,12443.0,12417.0,12783.0,12736.0
Store_5,9742.0,NaN,NaN,9170.0,9509.0
Store_6,8422.0,NaN,8582.0,9099.0,8967.0
Store_7,11763.0,12612.0,11468.0,11512.0,12958.0
Store_8,8343.0,NaN,8617.0,9183.0,9357.0


In [9]:
sales.isnull().sum()

Mon_Sales    0
Tue_Sales    3
Wed_Sales    2
Thu_Sales    0
Fri_Sales    0
dtype: int64

Five weekday sales columns for the same 8 stores - unlike `Gender`/`Married`/`Property_Area` in `loans.csv`, these columns really are comparable, same-unit measurements of the same thing.

In [10]:
sales.ffill(axis=1)

,Mon_Sales,Tue_Sales,Wed_Sales,Thu_Sales,Fri_Sales
Store_ID,,,,,
Store_1,7643.0,7623.0,8217.0,7670.0,7913.0
Store_2,13102.0,13203.0,13203.0,12921.0,12086.0
Store_3,11210.0,11376.0,10456.0,10688.0,10593.0
Store_4,13055.0,12443.0,12417.0,12783.0,12736.0
Store_5,9742.0,9742.0,9742.0,9170.0,9509.0
Store_6,8422.0,8422.0,8582.0,9099.0,8967.0
Store_7,11763.0,12612.0,11468.0,11512.0,12958.0
Store_8,8343.0,8343.0,8617.0,9183.0,9357.0


`axis=1` here means "fill left-to-right, using this **same row's** other columns," not "use other rows in the same column" (that's `axis=0`, the default everywhere else in this repo's notes). `Store_5` is missing both Tuesday and Wednesday - both get filled with Monday's figure, a defensible placeholder ("we don't have this week's number yet, use last known") for a live sales dashboard, in a way that would never make sense for `loans.csv`'s unrelated columns.

In [11]:
sales.mean(axis=1)

Store_ID
Store_1     7813.200000
Store_2    12828.000000
Store_3    10864.600000
Store_4    12686.800000
Store_5     9473.666667
Store_6     8767.500000
Store_7    12062.600000
Store_8     8875.000000
dtype: float64

A second, genuinely meaningful `axis=1` operation: each store's **average daily sales** across the (now-filled) week - a row-wise reduction that only makes sense because the five columns are truly the same kind of quantity.

## Recap

- `ffill`/`bfill` are only correct when row order carries real meaning - the ordered `daily_temperature.csv` is where they belong; `loans.csv` is where they merely *run*.
- `axis=1` means "across this row's other columns" - only meaningful when those columns are genuinely comparable, like `store_sales_wide.csv`'s Mon-Fri figures, never `loans.csv`'s unrelated `Gender`/`Married`/`Property_Area`.
- Full explanation in `07_Filling_Missing_Values.MD`; the statistic-fill (mean/median/mode) walkthrough for `loans.csv` lives in `07_Filling_Missing_Values.ipynb`.